# Import

In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
from datetime import datetime
import os
import pytz
import json
import seaborn as sns
import ast
from scipy.stats import gmean
import math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Function Definition

In [1]:
# Covert string to list
def tolist(str):
        json_str = str.replace("'", '"')
        parsed = json.loads(json_str)
        return [[float(x) for x in sublist] for sublist in parsed]
# Find Median of a list
def median(x):
    L=list(x)
    sort=sorted(L)
    if len(sort) % 2 != 0:
        index=int((len(sort)+1)/2-1)
        n=sort[index]
    if len(sort) % 2 == 0:
        index=int(len(sort)/2)
        index1=index-1
        n=(sort[index]+sort[index1])/2
    return n
# Weighted Median
def weightedmidian(x):
    D=[]
    number=0
    price=0
    for l in x:
        number+=l[1]
        D.append([number,l[0]])
    if number % 2 != 0:
        index=int((number+1)/2)
    if number % 2 == 0:
        index=int(number/2)
    if index<=D[0][0]:
        price+=D[0][1]
    elif index>D[0][0]:
        for n in range(1,len(D)-1):
            if index==D[n][0] or D[n-1][0]<index<D[n][0]:
                price+=D[n][1]
    return price
# Weighted Arithmetic Mean
def weightedArithmetic(x):
    total_price=0
    total_quantity=0
    for l in x:
        price=l[0]*l[1]
        total_price+=price
        total_quantity+=l[1]
    if total_quantity !=0:
        mean=total_price/total_quantity
    else:
        mean=np.nan
    return mean
# WeightedGeometric
def weightedGeometric(x):
    l1=[]
    l2=[]
    for l in x:
        l1.append(l[0])
        l2.append(l[1])
    if sum(l2)!=0:
        mean=gmean(l1,weights=l2) 
    else:
        mean=np.nan
    return mean
# Weighted Variance
def weightedstandarddeviation(x):
    data=[]
    weight=[]
    for l in x:
        data.append(l[0])
        weight.append(l[1])
    data1 = np.asarray(data)
    weights = np.asarray(weight)
    if sum(weights)!=0:
         average = np.average(data1, weights=weights)
         variance = np.average((data1-average)**2, weights=weights)
    else:
        return np.nan
    return math.sqrt(variance)

# Extract

In [3]:
## Extract
# 1. Extract Data
dfs = pd.read_csv('MarketDepthDataLisa.csv')
dfs['Time']=dfs['Time'].astype(int)

# Transform and Save to csv

In [ ]:
## Transform
# 1.Check missing values of columns
columns_with_missing_values = dfs.columns[dfs.isna().any()]
print("Columns with missing values:", columns_with_missing_values.tolist())
# 2.Change Datatype (a. str to list b.str to float)
if dfs['BidData'].apply(lambda x: isinstance(x, list)).any()==False:
    dfs['BidData']=dfs['BidData'].apply(tolist)
if dfs['AskData'].apply(lambda x: isinstance(x, list)).any()==False:
    dfs['AskData']=dfs['AskData'].apply(tolist)
# 3.Add columns(features) Add features that reflects volatility and returns paper(https://www.sciencedirect.com/science/article/pii/S1544612321001124#sec0008)
dfs['Datetime'] = pd.to_datetime(dfs['Time'], unit='ms')
dfs['BidNumber']=dfs['BidData'].str.len()
dfs['AskNumber']=dfs['AskData'].str.len()
dfs['BestBidPrice']=dfs['BidData'].apply(lambda row: max(x[0] for x in row) 
                                         if isinstance(row, list) and row and all(isinstance(sublist, list) 
                                        for sublist in row) else np.nan)
dfs['BestAskPrice']=dfs['AskData'].apply(lambda row: min(x[0] for x in row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['BestBidQuantity']=dfs['BidData'].apply(lambda row: next((x[1] for x in row if x[0] == max(y[0] for y in row)), np.nan) 
                                            if isinstance(row, list) and row and all(isinstance(sublist, list) 
                                            for sublist in row) else np.nan)
dfs['BestAskQuantity']=dfs['AskData'].apply(lambda row: next((x[1] for x in row if x[0] == max(y[0] for y in row)), np.nan) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['Spread']=dfs['BestAskPrice']-dfs['BestBidPrice']
dfs['MidBidPrice']=dfs['BidData'].apply(lambda row: median(x[0] for x in row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['MidAskPrice']=dfs['AskData'].apply(lambda row: median(x[0] for x in row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['MidAskQuantity']=dfs['AskData'].apply(lambda row: next((x[1] for x in row if x[0] == median(y[0] for y in row)), np.nan) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['MidBidQuantity']=dfs['BidData'].apply(lambda row: next((x[1] for x in row if x[0] == median(y[0] for y in row)), np.nan) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['WeightedMidBidPrice']=dfs['BidData'].apply(lambda row: weightedmidian(row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['WeightedMidAskPrice']=dfs['AskData'].apply(lambda row: weightedmidian(row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['TotalBidQuantity']=dfs['BidData'].apply(lambda row: sum(x[1] for x in row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['TotalAskQuantity']=dfs['AskData'].apply(lambda row: sum(x[1] for x in row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['WeightedAverageBidPriceArithmetic']=dfs['BidData'].apply(lambda row: weightedArithmetic(row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['WeightedAverageAskPriceArithmetic']=dfs['AskData'].apply(lambda row: weightedArithmetic(row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['GeometricMeanBidPrice']=dfs['BidData'].apply(lambda row: weightedGeometric(row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['GeometricMeanAskPrice']=dfs['AskData'].apply(lambda row: weightedGeometric(row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['STDBid']=dfs['BidData'].apply(lambda row: weightedstandarddeviation(row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['STDAsk']=dfs['AskData'].apply(lambda row: weightedstandarddeviation(row) if isinstance(row, list) and row and all(isinstance(sublist, list) for sublist in row) else np.nan)
dfs['volatility'] = dfs['Spread'].rolling(window=3).std()
dfs['volatility'] = dfs['volatility'].fillna(dfs['volatility'].mean())
dfs.to_csv('ProcessedOrderLisa.csv', index=False)